# HYG/LQD 分红事件来源边界修复

首轮 HYG 历史 4226 行价格严格相同、仅最新前沿收盘改动 0.005，失败结果保留。冻结修复排除该最新前沿行，只接收过去精确相同的价格和分红事件，然后只提交原计划 LQD URL 一次。

In [1]:
import json,hashlib,subprocess,tarfile,tempfile,shlex,math
from pathlib import Path
from datetime import datetime,timezone,date
from zoneinfo import ZoneInfo
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_yahoo_credit_etf_history_source_20261023';plan=json.loads((card/'event_supplement_repair_plan.json').read_text());assert plan['frozen'] and not (card/'event_repair_attempt_started.json').exists()
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
raw=root/'data/snapshots/stage1_yahoo_credit_etf_history_source_20261023';eventroot=raw/'events'
checks=[]
def check(symbol,path):
 original=json.loads((raw/(symbol+'.json')).read_text())['chart']['result'][0];z=json.loads(path.read_text())['chart']['result'][0]
 assert z['meta']['symbol']==symbol and z['timestamp']==original['timestamp'] and len(z['timestamp'])==4227
 x=original['indicators']['quote'][0]['close'];y=z['indicators']['quote'][0]['close'];assert x[:-1]==y[:-1]
 last=datetime.fromtimestamp(z['timestamp'][-1],timezone.utc).astimezone(ZoneInfo('America/New_York')).date();assert last==date(2026,10,22) and last>date(2026,10,7)
 events=z.get('events') or {};div=events.get('dividends') or {};splits=events.get('splits') or {};assert len(div)>100
 ddates=[datetime.fromtimestamp(int(k),timezone.utc).astimezone(ZoneInfo('America/New_York')).date() for k in div];assert len(ddates)==len(set(ddates)) and all(math.isfinite(float(v['amount'])) and float(v['amount'])>0 for v in div.values())
 return {'symbol':symbol,'status':'all_pre_frontier_4226_close_rows_exact_and_dividend_events_received','raw_sha256':sha(path),'timestamps':len(z['timestamp']),'historical_close_rows_exact':len(x)-1,'frontier_date':str(last),'frontier_old_close':x[-1],'frontier_new_close':y[-1],'dividend_events':len(div),'first_dividend_date':str(min(ddates)),'last_dividend_date':str(max(ddates)),'split_events':len(splits),'historical_event_vintage_qualified':False}
hyg=eventroot/'HYG.json';checks.append(check('HYG',hyg))
state={'status':'repair_started','additional_GET_attempts':1,'prior_HYG_event_GETs':1,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'event_repair_attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
url=plan['remaining_exact_LQD_URL'];body=eventroot/'LQD.json';headers=eventroot/'LQD.headers';started=datetime.now(timezone.utc).isoformat();r=json.loads((card/'event_supplement_plan.json').read_text())['request']
cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(r['connect_timeout_seconds']),'--max-time',str(r['total_timeout_seconds_each']),'--max-filesize',str(r['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',r['user_agent'],'-w','%{http_code} %{url_effective}',url]
try:
 run=subprocess.run(cmd,capture_output=True,text=True,timeout=75);fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
 receipt={'symbol':'LQD','url':url,'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'stderr':run.stderr[:300]}
except Exception as exc:receipt={'symbol':'LQD','url':url,'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
(card/'event_repair_capture_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
failure=None
if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists() or body.stat().st_size>r['max_response_bytes_each']:failure='LQD_transport_or_size_failure'
else:
 try:checks.append(check('LQD',body))
 except Exception as exc:failure='LQD_identity_or_historical_overlap_failure';checks.append({'symbol':'LQD','status':failure,'error':type(exc).__name__+':'+str(exc)[:150],'raw_sha256':receipt['raw_sha256']})
files=[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(eventroot.iterdir()) if p.is_file()];manifest={'repair_plan_sha256':sha(card/'event_supplement_repair_plan.json'),'files':files,'LQD_receipt_sha256':sha(card/'event_repair_capture_receipt.json')};(card/'event_repair_source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='credit-event-repair-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for x in files:out.add(eventroot/x['name'],arcname='raw/'+x['name'])
  out.add(card/'event_repair_source_manifest.json',arcname='event_repair_source_manifest.json');out.add(card/'event_repair_capture_receipt.json',arcname='event_repair_capture_receipt.json')
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='credit-event-repair-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   for x in files:
    member=source.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
result={'status':'HYG_LQD_retrospective_dividend_event_sources_qualified_for_full_span_PIT_pending' if failure is None and len(checks)==2 else 'event_source_repair_partial_or_failed','failure':failure,'total_event_GETs':2,'checks':checks,'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'historical_first_public_event_vintage_qualified':False}
(card/'event_repair_result.json').write_text(json.dumps(result,indent=2)+'\n');print({'status':result['status'],'failure':failure,'checks':checks,'NAS_sha256':digest})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'HYG_LQD_retrospective_dividend_event_sources_qualified_for_full_span_PIT_pending', 'failure': None, 'checks': [{'symbol': 'HYG', 'status': 'all_pre_frontier_4226_close_rows_exact_and_dividend_events_received', 'raw_sha256': '6259b766f7b89c47af1ffebd1e6bdc1d3aabc39c81c81b8690be95e3b9a280f6', 'timestamps': 4227, 'historical_close_rows_exact': 4226, 'frontier_date': '2026-10-22', 'frontier_old_close': 77.07499694824219, 'frontier_new_close': 77.08000183105469, 'dividend_events': 200, 'first_dividend_date': '2010-02-01', 'last_dividend_date': '2026-10-01', 'split_events': 0, 'historical_event_vintage_qualified': False}, {'symbol': 'LQD', 'status': 'all_pre_frontier_4226_close_rows_exact_and_dividend_events_received', 'raw_sha256': '27166ce97e6b2e281eaa35d00ffd8a003c6a89bacd8cef9cb54c8c7215d8da7a', 'timestamps': 4227, 'historical_close_rows_exact': 4226, 'frontier_date': '2026-10-22', 'frontier_old_close': 104.30500030517578, 'frontier_new_close': 104.2750015258789, 'dividend_ev

## 中文结论

HYG 与 LQD 每只基金 200 条正的分红事件、0 拆股。两次请求日期完全相同，除 2026-10-22 当前可变前沿行外 4,226 个历史 close 逐值相同；前沿差异和首轮 HYG 失败保留。原事件字节 NAS SHA-256 `510e0b0be5abc043163e085ab8989718507a1f64ce93de2ff1272b420ef92fbb` 限定恢复通过。没有历史首次分红版本资格，0 模型/账户。